In [1]:
import pandas as pd

df = pd.read_csv('resumes_processed.csv')
print(df.shape)
df.head()

(2484, 2)


,cleaned_resume,category
0,hr administratormarketing associate hr adminis...,HR
1,hr specialist us hr operations summary versati...,HR
2,hr director summary years experience recruitin...,HR
3,hr specialist summary dedicated driven dynamic...,HR
4,hr manager skill highlights hr skills hr depar...,HR


In [3]:
# Check how many NaN values
print("NaN values:", df['cleaned_resume'].isnull().sum())

# Drop them
df = df.dropna(subset=['cleaned_resume'])
print("Shape after dropping NaN:", df.shape)

NaN values: 1
Shape after dropping NaN: (2483, 2)


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df['cleaned_resume'])

print("Matrix shape:", tfidf_matrix.shape)

Matrix shape: (2483, 5000)


In [5]:
feature_names = vectorizer.get_feature_names_out()
print("Total features:", len(feature_names))
print("\nFirst 20 words:", feature_names[:20])
print("\nLast 20 words:", feature_names[-20:])

Total features: 5000

First 20 words: ['aa' 'aas' 'ab' 'abc' 'abilities' 'ability' 'able' 'abreast' 'abroad'
 'absence' 'abuse' 'academic' 'academy' 'accelerated' 'accept'
 'acceptable' 'acceptance' 'accepted' 'access' 'accessories']

Last 20 words: ['xerox' 'xml' 'xp' 'year' 'yearend' 'yearly' 'years' 'yes' 'yet' 'yield'
 'ymca' 'yoga' 'york' 'young' 'youth' 'youtube' 'yrs' 'zero' 'zone'
 'zumba']


In [6]:
import re
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def match_resume_to_jd(resume_text, jd_text):
    # Clean both texts
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    # Vectorize both together
    tfidf = vectorizer.transform([cleaned_resume, cleaned_jd])
    
    # Compute similarity
    score = cosine_similarity(tfidf[0], tfidf[1])[0][0]
    
    return round(score * 100, 2)

In [8]:
sample_jd = """
We are looking for a Python Developer with 2+ years of experience.
Requirements: Python, Machine Learning, scikit-learn, pandas, SQL, REST APIs.
Experience with data preprocessing and model deployment is a plus.
Strong problem solving and communication skills required.
"""

# Pick a random resume from dataset and test
sample_resume = df['cleaned_resume'][0]

score = match_resume_to_jd(sample_resume, sample_jd)
print(f"Match Score: {score}%")
print(f"Resume Category: {df['category'][0]}")

Match Score: 1.52%
Resume Category: HR


In [14]:
# Test same JD against resumes from different categories
jd = """
Python Developer with experience in machine learning, data analysis,
scikit-learn, pandas, numpy, SQL, REST APIs and model deployment.
"""

print("Testing same Python Developer JD across different resume categories:\n")

categories_to_test = ['PYTHON', 'INFORMATION-TECHNOLOGY', 'HR', 'ACCOUNTANT', 'CHEF','ENGINEERING','CONSULTING']

for category in categories_to_test:
    # Get first resume from each category
    cat_resumes = df[df['category'] == category]
    if len(cat_resumes) > 0:
        resume = cat_resumes.iloc[0]['cleaned_resume']
        score = match_resume_to_jd(resume, jd)
        print(f"{category:15} → {score}%")

Testing same Python Developer JD across different resume categories:

INFORMATION-TECHNOLOGY → 6.22%
HR              → 1.42%
ACCOUNTANT      → 0.94%
CHEF            → 0.43%
ENGINEERING     → 1.88%


In [11]:
print(df['category'].unique())

<StringArray>
[                    'HR',               'DESIGNER', 'INFORMATION-TECHNOLOGY',
                'TEACHER',               'ADVOCATE',   'BUSINESS-DEVELOPMENT',
             'HEALTHCARE',                'FITNESS',            'AGRICULTURE',
                    'BPO',                  'SALES',             'CONSULTANT',
          'DIGITAL-MEDIA',             'AUTOMOBILE',                   'CHEF',
                'FINANCE',                'APPAREL',            'ENGINEERING',
             'ACCOUNTANT',           'CONSTRUCTION',       'PUBLIC-RELATIONS',
                'BANKING',                   'ARTS',               'AVIATION']
Length: 24, dtype: str


In [15]:
def get_missing_skills(resume_text, jd_text):
    # Clean both
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    # Get unique words from each
    resume_words = set(cleaned_resume.split())
    jd_words = set(cleaned_jd.split())
    
    # Words in JD but not in resume = missing skills
    missing = jd_words - resume_words
    
    # Remove very short words (less than 3 characters)
    missing = [w for w in missing if len(w) >= 3]
    
    return sorted(missing)

# Test it
sample_jd = """
Python Developer with experience in machine learning, scikit-learn, 
pandas, numpy, SQL, REST APIs, docker, git, model deployment, tensorflow.
"""

sample_resume = df[df['category'] == 'HR'].iloc[0]['cleaned_resume']
missing = get_missing_skills(sample_resume, sample_jd)

print("Skills in JD but missing from resume:")
print(missing)

Skills in JD but missing from resume:
['apis', 'deployment', 'developer', 'docker', 'git', 'learning', 'machine', 'model', 'numpy', 'pandas', 'python', 'rest', 'scikitlearn', 'sql', 'tensorflow']


In [16]:
def analyze_resume(resume_text, jd_text):
    score = match_resume_to_jd(resume_text, jd_text)
    missing = get_missing_skills(resume_text, jd_text)
    
    print(f"Match Score: {score}%")
    print(f"\nMissing Skills ({len(missing)} found):")
    for skill in missing:
        print(f"  - {skill}")

# Test it
sample_jd = """
Looking for a Python Developer with experience in machine learning, 
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
"""

it_resume = df[df['category'] == 'INFORMATION-TECHNOLOGY'].iloc[0]['cleaned_resume']
analyze_resume(it_resume, sample_jd)

Match Score: 0.18%

Missing Skills (14 found):
  - apis
  - developer
  - docker
  - git
  - learning
  - looking
  - machine
  - numpy
  - pandas
  - python
  - rest
  - scikitlearn
  - sql
  - tensorflow


In [19]:
# Filter only IT resumes for a more focused vectorizer
it_resumes = df[df['category'] == 'INFORMATION-TECHNOLOGY']['cleaned_resume']

vectorizer = TfidfVectorizer(max_features=5000)
vectorizer.fit(it_resumes)

print("Vectorizer retrained on IT resumes only")
print("Sample IT vocab:", vectorizer.get_feature_names_out()[:20])

Vectorizer retrained on IT resumes only
Sample IT vocab: ['aa' 'aas' 'abilities' 'ability' 'able' 'abreast' 'absence' 'abuja'
 'academic' 'academics' 'academicsbusiness' 'academy' 'accept'
 'acceptance' 'accepted' 'accepting' 'access' 'accessories' 'accomplish'
 'accomplished']


In [20]:
it_resume = df[df['category'] == 'INFORMATION-TECHNOLOGY'].iloc[0]['cleaned_resume']

sample_jd = """
Looking for a Python Developer with experience in machine learning, 
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
"""

analyze_resume(it_resume, sample_jd)

Match Score: 0.21%

Missing Skills (14 found):
  - apis
  - developer
  - docker
  - git
  - learning
  - looking
  - machine
  - numpy
  - pandas
  - python
  - rest
  - scikitlearn
  - sql
  - tensorflow


In [21]:
sample_jd = """
Looking for a Python Developer with experience in machine learning, 
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
"""

cleaned_jd = clean_text(sample_jd)
print("Cleaned JD words:", cleaned_jd.split())

# Check which JD words are actually in our vocabulary
vocab = vectorizer.get_feature_names_out()
jd_words = cleaned_jd.split()
found = [w for w in jd_words if w in vocab]
missing_from_vocab = [w for w in jd_words if w not in vocab]

print(f"\nJD words found in vocabulary: {found}")
print(f"JD words NOT in vocabulary: {missing_from_vocab}")

Cleaned JD words: ['looking', 'python', 'developer', 'experience', 'machine', 'learning', 'scikitlearn', 'pandas', 'numpy', 'sql', 'rest', 'apis', 'docker', 'git', 'tensorflow']

JD words found in vocabulary: ['looking', 'python', 'developer', 'experience', 'machine', 'learning', 'sql', 'rest', 'apis', 'git']
JD words NOT in vocabulary: ['scikitlearn', 'pandas', 'numpy', 'docker', 'tensorflow']


In [22]:
def get_missing_skills(resume_text, jd_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    resume_words = set(cleaned_resume.split())
    jd_words = set(cleaned_jd.split())
    
    # Remove stopwords and short words
    missing = jd_words - resume_words
    missing = [w for w in missing if len(w) >= 4]
    
    # Remove generic words that aren't really skills
    generic = {'looking', 'experience', 'developer', 'working', 'strong', 
               'ability', 'knowledge', 'years', 'good', 'must', 'will'}
    missing = [w for w in missing if w not in generic]
    
    return sorted(missing)

# Test
it_resume = df[df['category'] == 'INFORMATION-TECHNOLOGY'].iloc[0]['cleaned_resume']
sample_jd = """
Looking for a Python Developer with experience in machine learning, 
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
"""

missing = get_missing_skills(it_resume, sample_jd)
print("Missing skills:")
print(missing)

Missing skills:
['apis', 'docker', 'learning', 'machine', 'numpy', 'pandas', 'python', 'rest', 'scikitlearn', 'tensorflow']


In [23]:
import pickle

# Save vectorizer for later use
with open('tfidf_vectorizer.pkl', 'w') as f:
    pass

pickle.dump(vectorizer, open('tfidf_vectorizer.pkl', 'wb'))
print("Vectorizer saved!")

Vectorizer saved!
